# Tracing the Spectrum

The spectrum of a point source on a long-slit frame is a narrow horizontal ridge: bright along
the dispersion axis, only a few pixels wide across the slit. Everything the rest of this book
does, from measuring the sky and collapsing the image to one dimension to deciding which pixels
belong to the object at all, needs to know *where that ridge is*, column by column.

That path is the **trace**, and finding it is the first real reduction step. `specreduce.tracing` offers three `Trace` classes for doing this, and the choice between them is really a choice about
how much you are willing to assume:

| Class | Assumption | Use when |
|---|---|---|
| `FlatTrace` | the trace is a horizontal line | the instrument is well aligned, or you need a quick look |
| `FitTrace` | the trace is a smooth low-order curve | almost always; this is the workhorse |
| `ArrayTrace` | nothing; you supply the positions | the shape defeats a polynomial, or the positions come from elsewhere |

This chapter works through all three on the science frame, and, more importantly, shows how
to tell whether the trace you got is any good. 

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from astropy.visualization import simple_norm
from specreduce.tracing import ArrayTrace, FitTrace, FlatTrace

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)

## Finding the source

Before fitting anything, we look at where the spectrum is. This is the full science frame, drawn with
the same logarithmic stretch as in Chapter 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ax.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
)
ax.set(xlabel="Dispersion [pix]", ylabel="Cross-dispersion [pix]");

The bright horizontal ridge is TrES-3, and the fainter ridge above it is a second star that fell
in the slit. At this scale both look perfectly straight, which is exactly the impression a trace
has to correct. A better first view is to collapse the frame along the dispersion axis, which
turns the 2D image into a rough estimate of the **cross-dispersion profile**: rough, because the
collapse ignores the curvature of the trace and smears a tilted or curved trace over several
rows. Even so, a point source appears as a narrow peak sitting on the broad plateau of the sky.

In [ ]:
profile = obj.data.mean(axis=1)
sky_level = np.median(profile)

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(rows, profile, lw=0.8)
ax.axhline(sky_level, c="0.6", lw=0.8, ls="--")
ax.text(
    365,
    sky_level * 1.15,
    f"sky $\\approx$ {sky_level:.0f} e$^-$",
    ha="right",
    size=8,
    c="0.4",
)
ax.annotate(
    "target\nrow 135",
    (135, profile[135]),
    (147, profile[135] * 0.45),
    size=8,
    va="center",
    arrowprops=dict(arrowstyle="-", lw=0.6, color="0.4"),
)

ax.set(
    yscale="log",
    xlim=(0, ny - 1),
    xlabel="Cross-dispersion [pix]",
    ylabel="Mean signal [e$^-$]",
);

Four sources stand out. The tall peak at **row 135** is TrES-3 itself, about fifty times the sky
level. The other three are stars that happened to fall in the 40-arcsecond slit: one near row
311 at roughly three percent of the target, one blended into the wing of TrES-3, and a fainter
one still near row 40. We ignore all three.

The peak is narrow: its full width at half maximum is about six pixels, the seeing disk as
sampled by this binning. Bear in mind that this is the width of the *averaged* profile. The
collapse folds the drift of the trace along the dispersion axis into it, so the profile in any
single column is somewhat narrower. Even so, **FWHM ≈ 6 pixels** is the yardstick for everything
that follows. A trace error of half a pixel is negligible against a six-pixel profile; an error
of four pixels is catastrophic. It also sets the scale for what "accurate enough" means: there is
no point chasing hundredths of a pixel when the source itself is six across.

Now the crucial question: does the source stay at row 135? Here are three cross-dispersion cuts,
taken near the left edge, the middle, and the right edge of the detector.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)

for x0 in (32, 512, 960):
    cut = obj.data[:, x0 - 16 : x0 + 16].mean(axis=1)
    ax.plot(
        rows, cut / cut[np.abs(rows - 135) <= 10].max(), lw=0.9, label=f"column {x0}"
    )

ax.axvline(135, c="0.6", lw=0.8, ls="--")
ax.legend(frameon=False)
ax.set(xlim=(122, 148), xlabel="Cross-dispersion [pix]", ylabel="Normalized signal");

The peak is not in the same place in the three cuts. It drifts by several pixels, comparable to
the width of the profile itself. A fixed row *cannot* follow it.

## `FlatTrace`: assuming a straight line

The simplest possible trace is a constant. `FlatTrace` takes the image and a single
cross-dispersion position, and produces a trace that sits at that row for every column.

In [ ]:
flat_trace = FlatTrace(obj, trace_pos=135)

print(type(flat_trace.trace), flat_trace.trace.shape)
print(flat_trace.trace[:5])

Every `Trace` object exposes the same thing: a `.trace` attribute holding one cross-dispersion
position per dispersion column. That uniform interface is what lets `Background` and the
extraction classes accept any of the three classes interchangeably.

Overplotted on the frame, a flat trace at row 135 looks plausible enough, until you stretch the
vertical axis. We draw only 40 rows around the source below, so the frame appears
about twenty-five times taller than it really is; the drift that was invisible in Chapter 1 is
obvious here.

In [ ]:
def show_trace(ax, traces, ylim=(115, 155), title=None):
    """Display the science frame zoomed on the source, with traces overplotted."""
    ax.imshow(
        obj.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
    )
    for tr, label, style in traces:
        ax.plot(np.asarray(tr), label=label, **{"lw": 1.0, **style})
    ax.set(
        xlim=(0, nx - 1),
        ylim=ylim,
        xlabel="Dispersion [pix]",
        ylabel="Cross-dispersion [pix]",
    )
    if title:
        ax.set_title(title, size=9)
    if any(label for _, label, _ in traces):
        ax.legend(frameon=False, loc="upper left", fontsize=8)


fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)
show_trace(ax, [(flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--"))]);

The ridge starts below the line, rises to meet it around the middle of the detector, and
falls away again to end well below it. `FlatTrace` is not wrong here so much as *insufficient*,
and it is worth being clear about when it is the right call anyway. If your spectrograph really
is well aligned, a flat trace is faster, has no fit to go wrong, and cannot be seduced by a
cosmic ray. It is also the sensible choice for arc frames, where the "source" fills the slit and
there is no ridge to follow at all. Chapter 6 uses exactly that.

## `FitTrace`: fitting a curve

`FitTrace` does the obvious thing, in three steps:

1. **Bin** the image into groups of columns along the dispersion axis, averaging each group.
   Binning trades resolution along the trace for signal-to-noise in each measurement.
2. **Find the peak** in each bin's cross-dispersion profile, using `max`, `centroid`, or
   `gaussian`.
3. **Fit a 1D model** through those peak positions and evaluate it at every column.

Each step has a parameter, and the defaults are not always what you want. The default model is
a `Polynomial1D` of degree 1: a straight line, free to tilt but not to bend. The three cuts in
the previous section showed the peak rising and then falling across the detector, and no line
can follow that. The default is not broken; it does exactly what a degree-1 polynomial can do,
which is not enough here. The question is how much more freedom to give it.

### Choosing the polynomial degree

There is no ground truth to compare against, so the test has to be one of **convergence**: we fit a
range of degrees and ask how much the trace *changes* when one more term is added. While the
model is missing real structure, each extra degree moves the trace by a substantial fraction of
a pixel. Once it has enough freedom, an extra term can only follow noise, and the change drops
to the level of the per-bin measurement error and stays there. The statistic below is the RMS
change, over all columns, between the trace fitted at degree $d$ and at degree $d+1$.

The model here is `Chebyshev1D` rather than `Polynomial1D`. Both describe the same family of
curves, but a plain power series in $x$ over a range of 0–1000 produces a badly conditioned fit
at high degree, and `astropy` will warn about it.

In [ ]:
degrees = range(1, 8)
fitted = {}
for degree in degrees:
    trace = FitTrace(
        obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(degree)
    )
    fitted[degree] = np.asarray(trace.trace)

change = {d: np.std(fitted[d + 1] - fitted[d]) for d in degrees[:-1]}

for d, c in change.items():
    print(
        f"degree {d} -> {d + 1}:  span {np.ptp(fitted[d]):.2f} pix,  "
        f"RMS change {c:.3f} pix"
    )

The table tells a clear story. Going from degree 1 to 2 moves the trace by 0.78 pixels RMS,
an eighth of the profile's FWHM: the trace is curved, and the quadratic term does nearly all the
work. Degrees 3 and 4 add corrections of about 0.05 pixels, and from degree 5 on the changes are
a few hundredths of a pixel or less, nothing against a six-pixel profile. The fit has converged.

We take degree 4. The general rule is to take the **lowest degree at which adding a term stops
changing the trace**, because every extra free parameter is another chance for the fit to chase
noise or to swing past the ends of the data where nothing constrains it.


In [ ]:
trace = FitTrace(
    obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
trace_y = np.asarray(trace.trace)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "FitTrace, degree 4", dict(c="C1")),
        (flat_trace.trace, "FlatTrace(135)", dict(c="C3", ls="--", lw=0.8)),
    ],
)

The fitted trace follows the ridge along the whole detector: it starts 1.7 pixels below the
flat line, touches it near the middle, and drops 3.6 pixels below it by the red end, a total
drift of 3.7 pixels.


## Choosing `peak_method`

`FitTrace` offers three ways to locate the peak within each bin's cross-dispersion profile:

- `max` takes the brightest pixel. It has no fit to fail and does not care about the shape of the
profile, which makes it the robust choice for faint, blended, or oddly shaped sources, and it is
the default. Its cost is that every measurement is a whole pixel, so the bin positions carry a
quantization error that only the polynomial fit across many bins averages away; with few bins
or a high-degree model the rounding is fitted rather than averaged out.

- `gaussian` fits a Gaussian on a constant baseline to each bin's profile and takes the fitted
center. On a bright, well-formed profile it is the most precise of the three, locating the peak
to a small fraction of a pixel, and it is what the fits above used. It is also the most
fragile: where the signal is weak or the profile is not Gaussian the per-bin fits become
unstable, and a fit whose center leaves the frame silently reuses the previous bin's answer.

- `centroid` takes the position that splits the bin's flux in half. It is cheap and sub-pixel,
but it is computed over the whole cross-dispersion range it is given, and the sky is part of
that flux. Without a `window`, the sky on both sides of the source and every other object on
the slit pull the centroid toward the middle of the slit, most strongly where the target is
faintest, and the trace leaves the source with no warning. Given a `window` narrow enough that
the target's own flux dominates the sum, it agrees closely with `gaussian`. Do not use
`centroid` without one.

The practical choice is `gaussian` by default on a bright, well-formed profile, and `max` when
speed matters or the profile is too ugly to fit.

### Confining the fit

Three arguments restrict where `FitTrace` looks, and they are worth setting even when a bright,
isolated target does not need them. `guess` tells `FitTrace` where to start instead of taking
the brightest row of the collapsed frame, and `window` confines the peak search in every bin to
`guess ± window` rows. Together they make the fit deterministic rather than dependent on
whatever happens to be brightest. `window` cuts both ways, though: it keeps the fit from
wandering onto a brighter neighbor, but it also caps how far the trace can move, and a source
that drifts further than `window` hits the edge of its own search region and flattens out. Set
it comfortably wider than the drift you expect, and narrower than the separation to the nearest
interloper.

`disp_bounds=(lo, hi)` confines the fit along the *dispersion* axis instead. The bins are laid
over that column range only, the model is fitted to the peaks found there, and it is then
evaluated at every column of the image, so outside the bounds the trace is an extrapolation. It
is the tool for a source whose signal fades along the trace, where the bins at the faint end
return peaks too poor to fit and would otherwise steer the polynomial. The extrapolated stretch
is a model assumption, sound for a smooth optical distortion and wrong for a kink or a flexure
jump, so draw it over the image before trusting it.

These arguments are what make a faint or crowded target tractable. `guess` and `window` select
the source and keep a brighter neighbor out of the fit, `disp_bounds` keeps the poorly measured
columns out, and fewer `bins` with a lower-degree model put more signal into each measurement
and leave the fit fewer points to be misled by. On a source only a few times the sky level,
expect the peak methods to disagree: `max` is the robust choice, and `gaussian` keeps its
precision only where the profile is well measured.

## `ArrayTrace`: supplying positions directly

Sometimes no polynomial will do. The trace may have a kink, the instrument may have flexed
mid-exposure, or the positions may come from somewhere else entirely. `ArrayTrace` takes the
positions verbatim.

It needs one value per dispersion column. The simplest demonstration is a round trip (hand it
the positions of the trace fitted above and get the same trace back), but the realistic use on
a dataset like this one is a time series: fit the trace once, on a bright exposure, and apply it
to every other frame through `ArrayTrace`, so that all of them are extracted along exactly the
same path. `ArrayTrace` also fixes up mismatched lengths rather than refusing them, and it is
worth seeing what that does:

In [ ]:
array_trace = ArrayTrace(obj, trace=trace_y)
print(f"round trip exact: {np.allclose(array_trace.trace, trace_y)}")

short_trace = ArrayTrace(obj, trace=trace_y[:900])
n_masked = np.ma.getmaskarray(short_trace.trace).sum()
print(
    f"900 positions in -> {short_trace.trace.shape[0]} out, {n_masked} of them masked"
)

An array longer than the image is truncated; a shorter one is padded at the end with masked
values, and non-finite entries are masked too. Convenient, but it means a length bug produces a
silently part-masked trace instead of an exception. Check `np.ma.getmaskarray(trace.trace).sum()`
if a trace behaves oddly.

`ArrayTrace` is the more flexible option and correspondingly the more dangerous one: with no
smooth model to enforce, whatever noise is in the input positions goes straight into the trace
and from there into the extracted spectrum. Use it when you genuinely need the freedom, and
smooth the input when you do.

## Shifting a trace

`Trace` objects support addition and subtraction, returning **shifted copies** displaced along
the cross-dispersion axis. The original is untouched.

In [ ]:
above = trace + 20
below = trace - 20

print(f"original  {np.asarray(trace.trace)[512]:.2f}")
print(f"+20       {np.asarray(above.trace)[512]:.2f}")
print(f"-20       {np.asarray(below.trace)[512]:.2f}")

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
show_trace(
    ax,
    [
        (trace_y, "trace", dict(c="C1")),
        (above.trace, "trace + 20", dict(c="C0", ls="--", lw=0.8)),
        (below.trace, "trace - 20", dict(c="C0", ls="--", lw=0.8)),
    ],
    ylim=(100, 170),
);

This is not a curiosity. It is how the next chapter places its sky windows: the background is
measured in bands that run *parallel to the trace*, offset either side of the source, so that
every pixel in a band is at the same distance from the object as the frame curves. Offsetting a
straight line instead would let the sky bands slide relative to the source, which is precisely
the error a fitted trace exists to avoid.

Positions that fall outside the image are masked automatically rather than raising, so a shift
that runs off the detector degrades gracefully.

## Summary

- A trace is one cross-dispersion position per dispersion column, exposed as `.trace` on all
  three classes. Whatever produced it, downstream operations consume it identically.
- On this frame the source moves **3.7 pixels** across the detector, against a profile FWHM of
  **6 pixels**, a drift a straight trace cannot follow.
- Validate with what you have: scan the degree and stop at the lowest one where adding a term no
  longer moves the trace (degree 4 here), and draw the result over the image.
- `peak_method` depends on the source. `gaussian` is the most precise on a bright, well-formed
  profile and the most fragile elsewhere; `max` is robust but quantized to whole pixels and
  relies on the fit to average that away; `centroid` silently wanders off the source unless
  given a `window`.
- `guess`, `window`, `disp_bounds` and `bins` confine the fit. On a faint or crowded source they
  select the right source, keep a brighter neighbor and the poorly measured columns out, and put
  more signal into each measurement; prefer `max` there.
- `trace ± delta` gives shifted copies, which is how background windows are placed.

Next: [Arc Lamps](05-arc-lamps.ipynb), the wavelength-reference frames, and the first place the
frame's tilt becomes impossible to ignore.